# Explainable Pneumonia Detection Under Computational Constraints
## Results Analysis Notebook

This notebook loads results from all four experiments and produces:
- Classification metrics comparison table
- Computational benchmark comparison table
- Accuracy vs. parameter count tradeoff plot
- AUC-ROC comparison bar chart
- Inference time comparison (GPU vs CPU)
- Confusion matrix grid
- Grad-CAM visualization grid

Run all cells after all four experiments have completed.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('muted')

RESULTS_DIR = Path('experiments/results')
MODELS = ['lightcxr', 'mobilenetv2', 'efficientnet_b0', 'resnet18']
MODEL_LABELS = {
    'lightcxr':        'LightCXR (ours)',
    'mobilenetv2':     'MobileNetV2',
    'efficientnet_b0': 'EfficientNet-B0',
    'resnet18':        'ResNet-18',
}
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

print('Setup complete.')

## 1. Load All Results

In [ ]:
def load_results(model_name):
    model_dir = RESULTS_DIR / model_name
    with open(model_dir / f'{model_name}_metrics.json') as f:
        metrics = json.load(f)
    with open(model_dir / f'{model_name}_benchmark.json') as f:
        benchmark = json.load(f)
    with open(model_dir / f'{model_name}_history.json') as f:
        history = json.load(f)
    return metrics, benchmark, history

all_metrics   = {}
all_benchmark = {}
all_history   = {}

for model in MODELS:
    try:
        m, b, h = load_results(model)
        all_metrics[model]   = m
        all_benchmark[model] = b
        all_history[model]   = h
        print(f'  Loaded: {model}')
    except FileNotFoundError:
        print(f'  Missing: {model} — run experiment first')

## 2. Classification Metrics Table

In [ ]:
metric_rows = []
for model in MODELS:
    if model not in all_metrics:
        continue
    m = all_metrics[model]
    metric_rows.append({
        'Model':       MODEL_LABELS[model],
        'Accuracy':    f"{m['accuracy']:.4f}",
        'F1':          f"{m['f1']:.4f}",
        'AUC-ROC':     f"{m['auc_roc']:.4f}",
        'Precision':   f"{m['precision']:.4f}",
        'Recall':      f"{m['recall']:.4f}",
        'Specificity': f"{m['specificity']:.4f}",
    })

metrics_df = pd.DataFrame(metric_rows).set_index('Model')
print('Classification Metrics — Test Set')
print('=' * 70)
print(metrics_df.to_string())
metrics_df

## 3. Computational Benchmark Table

In [ ]:
bench_rows = []
for model in MODELS:
    if model not in all_benchmark:
        continue
    b = all_benchmark[model]
    bench_rows.append({
        'Model':          MODEL_LABELS[model],
        'Params (M)':     f"{b['trainable_params']/1e6:.3f}",
        'FLOPs (M)':      f"{b['flops']/1e6:.1f}",
        'GPU (ms)':       f"{b['gpu_mean_ms']:.2f}",
        'CPU (ms)':       f"{b['cpu_mean_ms']:.2f}",
        'Train Time (s)': f"{b.get('train_duration_seconds', 'N/A')}",
    })

bench_df = pd.DataFrame(bench_rows).set_index('Model')
print('Computational Benchmark')
print('=' * 70)
print(bench_df.to_string())
bench_df

## 4. Accuracy vs. Parameter Count (Efficiency Tradeoff Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i, model in enumerate(MODELS):
    if model not in all_metrics:
        continue
    params_m = all_benchmark[model]['trainable_params'] / 1e6
    auc      = all_metrics[model]['auc_roc']
    ax.scatter(params_m, auc, s=180, color=COLORS[i], zorder=5, label=MODEL_LABELS[model])
    ax.annotate(
        MODEL_LABELS[model],
        (params_m, auc),
        textcoords='offset points',
        xytext=(8, 4),
        fontsize=9,
        color=COLORS[i],
    )

ax.set_xlabel('Trainable Parameters (Millions)', fontsize=11)
ax.set_ylabel('AUC-ROC (Test Set)', fontsize=11)
ax.set_title('Accuracy–Efficiency Tradeoff', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_tradeoff.png')

## 5. AUC-ROC Comparison Bar Chart

In [ ]:
available = [m for m in MODELS if m in all_metrics]
labels    = [MODEL_LABELS[m] for m in available]
auc_vals  = [all_metrics[m]['auc_roc'] for m in available]
f1_vals   = [all_metrics[m]['f1'] for m in available]
acc_vals  = [all_metrics[m]['accuracy'] for m in available]

x = np.arange(len(available))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, auc_vals, width, label='AUC-ROC', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x,         f1_vals,  width, label='F1 Score', color='#4CAF50', alpha=0.85)
bars3 = ax.bar(x + width, acc_vals, width, label='Accuracy', color='#FF9800', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=7.5
        )

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Classification Metrics by Architecture', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_metrics_comparison.png')

## 6. Inference Time Comparison (GPU vs CPU)

In [ ]:
available = [m for m in MODELS if m in all_benchmark]
labels    = [MODEL_LABELS[m] for m in available]
gpu_times = [all_benchmark[m]['gpu_mean_ms'] for m in available]
cpu_times = [all_benchmark[m]['cpu_mean_ms'] for m in available]

x = np.arange(len(available))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, gpu_times, width, label='GPU', color='#3F51B5', alpha=0.85)
ax.bar(x + width/2, cpu_times, width, label='CPU', color='#F44336', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Inference Time (ms)', fontsize=11)
ax.set_title('Single-Sample Inference Latency', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_inference_time.png')

## 7. Confusion Matrix Grid

In [ ]:
available = [m for m in MODELS if m in all_metrics]
fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))

if len(available) == 1:
    axes = [axes]

for ax, model in zip(axes, available):
    m = all_metrics[model]
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['NORMAL', 'PNEUMONIA'],
        yticklabels=['NORMAL', 'PNEUMONIA'],
        ax=ax, cbar=False,
    )
    ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_confusion_matrices.png')

## 8. Training Curves

In [ ]:
available = [m for m in MODELS if m in all_history]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, model in enumerate(available):
    h = all_history[model]
    epochs = range(1, len(h['train_loss']) + 1)
    axes[0].plot(epochs, h['train_loss'], label=MODEL_LABELS[model], color=COLORS[i], linewidth=1.5)
    axes[0].plot(epochs, h['val_loss'],   linestyle='--', color=COLORS[i], alpha=0.6)
    axes[1].plot(epochs, h['val_acc'],    label=MODEL_LABELS[model], color=COLORS[i], linewidth=1.5)

axes[0].set_title('Loss Curves (solid=train, dashed=val)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, linestyle='--')

axes[1].set_title('Validation Accuracy', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_training_curves.png')

## 9. Grad-CAM Visualization Grid

In [ ]:
available = [m for m in MODELS if (RESULTS_DIR / m / f'{m}_gradcam.png').exists()]

fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
if len(available) == 1:
    axes = [axes]

for ax, model in zip(axes, available):
    img = Image.open(RESULTS_DIR / model / f'{model}_gradcam.png')
    # Crop to overlay panel only (rightmost third)
    w, h = img.size
    overlay = img.crop((2 * w // 3, 0, w, h))
    ax.imshow(overlay)
    ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Grad-CAM Overlays — Pneumonia Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plot_gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/plot_gradcam_grid.png')

## 10. Summary — Key Findings

In [ ]:
available = [m for m in MODELS if m in all_metrics]

# Best model by AUC-ROC
best_auc   = max(available, key=lambda m: all_metrics[m]['auc_roc'])
# Most efficient model (best AUC-ROC per million params)
best_eff   = max(available, key=lambda m: all_metrics[m]['auc_roc'] / (all_benchmark[m]['trainable_params'] / 1e6))
# Fastest CPU inference
fastest    = min(available, key=lambda m: all_benchmark[m]['cpu_mean_ms'])

print('Key Findings')
print('=' * 50)
print(f'  Best AUC-ROC:         {MODEL_LABELS[best_auc]} ({all_metrics[best_auc]["auc_roc"]:.4f})')
print(f'  Most efficient:       {MODEL_LABELS[best_eff]} ({all_metrics[best_eff]["auc_roc"]:.4f} AUC / {all_benchmark[best_eff]["trainable_params"]/1e6:.3f}M params)')
print(f'  Fastest CPU inference:{MODEL_LABELS[fastest]} ({all_benchmark[fastest]["cpu_mean_ms"]:.2f}ms)')
print()
print('Full results saved to: experiments/results/results_summary.csv')